# MLflow Demo — Predictive Maintenance Project

This notebook shows how to use **MLflow** to track our XGBoost model experiments.

**What we'll do:**
1. Load the same data our project uses
2. Train the model and log everything with MLflow
3. Run 3 experiments with different settings
4. See how MLflow helps us compare results

**Note:** This is a demo — we're not changing the actual project code.

## Step 1 — Install MLflow

In [ ]:
!pip install mlflow xgboost scikit-learn pandas numpy

## Step 2 — Import Libraries

In [ ]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
import joblib

print("MLflow version:", mlflow.__version__)

## Step 3 — Load Our Project's Data

We use the same model files from `ai/model/`
- `xgboost_rul_model.pkl` — our trained model
- `standard_scaler.pkl` — the scaler
- `feature_names.pkl` — feature names

In [ ]:
MODEL_DIR = Path("../ai/model")

# Load the existing model and tools
existing_model = joblib.load(MODEL_DIR / "xgboost_rul_model.pkl")
scaler = joblib.load(MODEL_DIR / "standard_scaler.pkl")
feature_names = joblib.load(MODEL_DIR / "feature_names.pkl")

print(f"Model type: {type(existing_model).__name__}")
print(f"Number of features: {len(feature_names)}")
print(f"Features: {feature_names[:5]} ... (and {len(feature_names)-5} more)")
print(f"Existing model params: {existing_model.get_params()}")

## Step 4 — Generate Training Data

We'll create simple synthetic data to demo MLflow. In the real project, this would come from C-MAPSS.

In [ ]:
np.random.seed(42)
n_samples = 1000
n_features = len(feature_names)

# Generate random sensor readings
X = np.random.randn(n_samples, n_features)

# Generate RUL values (what we want to predict)
w = np.random.randn(n_features) * 0.5
y = X @ w + np.random.randn(n_samples) * 10 + 125
y = np.clip(y, 0, 300)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"RUL range: {y.min():.1f} to {y.max():.1f}")

## Step 5 — Train One Model with MLflow

This is the basic idea:
1. Start a run
2. Set parameters
3. Train
4. Log metrics
5. Save the model

In [ ]:
# Set the experiment name
mlflow.set_experiment("Predictive Maintenance - Demo")

# Start a run
with mlflow.start_run(run_name="First Training Run"):
    
    # 1. Set parameters
    params = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "objective": "reg:squarederror"
    }
    mlflow.log_params(params)
    
    # 2. Train the model
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    
    # 3. Make predictions
    y_pred = model.predict(X_test)
    
    # 4. Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R2:   {r2:.2f}")
    
    # 5. Log metrics to MLflow
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    # 6. Save the model
    mlflow.xgboost.log_model(model, "model")
    
    print("\nDone! Check MLflow UI to see this run.")

## Step 6 — Run 3 More Experiments

Let's try different settings and see how MLflow helps us compare.

In [ ]:
# Experiment 1: More trees, deeper
with mlflow.start_run(run_name="More Trees, Deeper"):
    params = {"n_estimators": 200, "max_depth": 8, "learning_rate": 0.1, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"Run 1 - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

# Experiment 2: Fewer trees, shallower
with mlflow.start_run(run_name="Fewer Trees, Shallow"):
    params = {"n_estimators": 50, "max_depth": 4, "learning_rate": 0.05, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"Run 2 - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

# Experiment 3: Default settings (our project's current model)
with mlflow.start_run(run_name="Default Settings"):
    params = {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"Run 3 - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

print("\nAll 3 experiments logged!")

## Step 7 — Log Batch Predictions

We can also log predictions to track how the model performs on new data.

In [ ]:
with mlflow.start_run(run_name="Batch Prediction Log"):
    
    # Use the last trained model to predict
    y_pred = model.predict(X_test)
    
    # Log prediction statistics
    mlflow.log_metric("avg_predicted_rul", float(np.mean(y_pred)))
    mlflow.log_metric("min_predicted_rul", float(np.min(y_pred)))
    mlflow.log_metric("max_predicted_rul", float(np.max(y_pred)))
    mlflow.log_metric("std_predicted_rul", float(np.std(y_pred)))
    
    # Count engines in each health category
    critical = int(np.sum(y_pred < 30))
    warning = int(np.sum((y_pred >= 30) & (y_pred < 75)))
    healthy = int(np.sum(y_pred >= 75))
    
    mlflow.log_metric("engines_critical", critical)
    mlflow.log_metric("engines_warning", warning)
    mlflow.log_metric("engines_healthy", healthy)
    
    print(f"Batch Prediction Summary:")
    print(f"  Average RUL: {np.mean(y_pred):.1f}")
    print(f"  Critical (<30): {critical} engines")
    print(f"  Warning (30-75): {warning} engines")
    print(f"  Healthy (>75): {healthy} engines")
    
    print("\nPrediction log saved to MLflow!")

## Step 8 — View Results

Run this command to open MLflow UI in your browser:

```bash
mlflow ui
```

Then open http://localhost:5000 to see all experiments.

**What you can do in the UI:**
- Compare all runs side by side
- See which parameters gave the best results
- Download saved models
- View metrics over time

In [ ]:
# Show all runs in this experiment
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Predictive Maintenance - Demo")

if experiment:
    runs = client.search_runs(experiment.experiment_id)
    print(f"Found {len(runs)} runs:\n")
    
    for i, run in enumerate(runs, 1):
        data = run.data
        print(f"Run {i}: {run.info.run_name}")
        print(f"  Params: n_estimators={data.params.get('n_estimators')}, max_depth={data.params.get('max_depth')}")
        rmse_val = data.metrics.get('rmse')
        r2_val = data.metrics.get('r2')
        rmse_str = f"{rmse_val:.2f}" if rmse_val is not None else "N/A"
        r2_str = f"{r2_val:.2f}" if r2_val is not None else "N/A"
        print(f"  RMSE: {rmse_str}, R2: {r2_str}")
        print()
else:
    print("No experiments found. Make sure you ran the cells above.")

## Step 9 — How to Add This to the Project

Here's what you would change in the actual project code:

In [ ]:
# Example: Training script (ai/train.py)
# ----------------------------------------
# import mlflow
# import mlflow.xgboost
#
# mlflow.set_experiment("Predictive Maintenance")
#
# with mlflow.start_run(run_name="xgboost_training"):
#     params = {"n_estimators": 100, "max_depth": 6}
#     mlflow.log_params(params)
#     
#     model = xgb.XGBRegressor(**params)
#     model.fit(X_train, y_train)
#     
#     y_pred = model.predict(X_test)
#     mlflow.log_metric("rmse", rmse)
#     mlflow.xgboost.log_model(model, "model")

# Example: Prediction service (application/prediction/service.py)
# --------------------------------------------------------------
# import mlflow
#
# with mlflow.start_run(run_name="batch_prediction"):
#     predictions = model.predict(X)
#     mlflow.log_metric("avg_rul", float(np.mean(predictions)))
#     mlflow.log_metric("critical_engines", int(np.sum(predictions < 30)))

print("These are the code snippets to add to the project.")
print("Don't forget to add 'mlflow' to requirements.txt!")

## Summary

**What we learned:**
1. MLflow tracks experiments automatically
2. We can compare different model settings
3. Models and metrics are saved for later
4. The UI makes it easy to see results

**Commands to remember:**
- `mlflow.set_experiment("name")` — create/select experiment
- `mlflow.start_run()` — start tracking
- `mlflow.log_params()` — save parameters
- `mlflow.log_metric()` — save results
- `mlflow.xgboost.log_model()` — save model
- `mlflow ui` — open dashboard